# Archery Tag Balance Lab — Colab setup

当前只检查仓库与配置，不运行比赛、不训练模型。CPU 即可。保存 Notebook 不会自动提交仓库中的文件。


## 1. 获取项目

公开仓库地址已预填，直接运行即可，无需 Token。此单元不会覆盖已有项目；若目录已存在，则保留当前版本，不自动拉取。

以后若改为私有仓库，参见 docs/colab_workflow.md。


In [ ]:
from pathlib import Path
import subprocess
from urllib.parse import urlparse

REPO_URL = "https://github.com/Echoudexigaigu/archery-tag-simulate.git"
PROJECT_DIR = Path("/content/archery-tag-simulate")

if PROJECT_DIR.exists():
    print("Existing project retained:", PROJECT_DIR)
else:
    parsed = urlparse(REPO_URL)
    if (parsed.scheme != "https" or parsed.hostname != "github.com"
            or parsed.netloc != "github.com" or parsed.query or parsed.fragment):
        raise ValueError("Set a credential-free HTTPS GitHub repository URL first.")
    parts = parsed.path.strip("/").split("/")
    if len(parts) != 2 or not all(parts):
        raise ValueError("Expected https://github.com/OWNER/REPO.git")
    import os
    clone_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    subprocess.run(["git", "clone", "--", REPO_URL, str(PROJECT_DIR)],
                   check=True, env=clone_env)


## 2. 检查配置与待测参数

所有 null 均为待测或待确认；不以示例值冒充现场参数。


In [ ]:
import json

config_path = PROJECT_DIR / "configs" / "baseline.json"
if not config_path.is_file():
    raise FileNotFoundError("Clone or extract the project into /content/archery-tag-simulate first.")
config = json.loads(config_path.read_text())
print("Status:", config["status"])
print("Round length:", config["time"]["round_seconds"], "seconds")
print("Racks:", config["rack"]["count"], "| spots:", config["rack"]["spot_count"])

def missing_fields(value, prefix=""):
    if isinstance(value, dict):
        for key, item in value.items():
            yield from missing_fields(item, f"{prefix}.{key}" if prefix else key)
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            yield from missing_fields(item, f"{prefix}[{idx}]")
    elif value is None:
        yield prefix

print("\nUnresolved fields:")
print("\n".join(missing_fields(config)))


## 3. 记录代码版本

ZIP 无 Git 历史时，手动记录下载版本。正式实验还应保存配置快照、依赖版本、种子和策略版本。


In [ ]:
import platform

print("Python:", platform.python_version())
if (PROJECT_DIR / ".git").exists():
    revision = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip()
    status = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True)
    print("Commit:", revision)
    print("Uncommitted changes:", bool(status.strip()))
else:
    print("No Git history: record the source commit separately.")


## 下一步

填写现场尺寸、靶架几何和待确认规则。当前没有比赛回放、胜率或 RL 训练。大型输出及未推送修改需另存到 Drive；Colab 工作目录不是持久备份。
